In [19]:
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, GATv2Conv
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
import matplotlib.pyplot as plt

In [20]:
name_data = "Cora"
dataset = Planetoid(root=f"./tmp/{name_data}", name=name_data)
dataset.transform =T.NormalizeFeatures()

print(f"Number of classes in {name_data}: {dataset.num_classes}")
print(f"Number of Node features in {name_data}: {dataset.num_node_features}")

Number of classes in Cora: 7
Number of Node features in Cora: 1433


In [21]:
import torch
import torch.nn.functional as F
class GAT(torch.nn.Module):
    def __init__(self):
        super(GAT, self).__init__()
        self.hid = 8
        self.in_head = 8
        self.out_head = 1
        
        self.conv1 = GATv2Conv(dataset.num_features, self.hid, heads=self.in_head, dropout=0.6)
        self.conv2 = GATv2Conv(self.hid*self.in_head, dataset.num_classes, concat=False, heads=self.out_head, dropout=0.6)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        
        return F.log_softmax(x, dim=1)

In [22]:
device = "cpu" 
model = GAT().to(device)
data = dataset[0].to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

In [23]:
model.train()
for epoch in range(1000):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    
    if epoch % 200 == 0:
        print(loss)
    loss.backward()
    optimizer.step()

tensor(1.9478, grad_fn=<NllLossBackward0>)
tensor(0.7085, grad_fn=<NllLossBackward0>)
tensor(0.6536, grad_fn=<NllLossBackward0>)
tensor(0.5479, grad_fn=<NllLossBackward0>)
tensor(0.4498, grad_fn=<NllLossBackward0>)


In [24]:
model.eval()
_, pred = model(data).max(dim=1)
correct = float(pred[data.test_mask].eq(data.y[data.test_mask]).sum().item())
acc = correct / data.test_mask.sum().item()
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.8190
